# 02. Feature Engineering

이 노트북은 `../data/raw/`의 원본 CSV를 읽어 첫 번째 재사용 가능 피처셋을 생성합니다. 결과물은 `../data/proceed/train_fe_v1.csv`, `../data/proceed/test_fe_v1.csv`로 저장하며, 이후 baseline/modeling 노트북은 이 파일을 입력으로 사용합니다.

v1 피처셋의 목표는 과도한 실험을 넣기보다 EDA에서 확인한 안전한 변환을 고정하는 것입니다.

- 헷갈리는 무게 컬럼 rename
- `Sex` one-hot encoding
- 물리적으로 해석 가능한 크기/비율 파생 피처 생성
- `Height == 0` indicator 추가
- train 기준 median으로 파생 피처 결측/무한대 처리
- train/test feature schema 검증 후 저장


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCEED_DIR = PROJECT_ROOT / "data" / "proceed"
PROCEED_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = RAW_DIR / "train.csv"
TEST_PATH = RAW_DIR / "test.csv"
TRAIN_OUTPUT_PATH = PROCEED_DIR / "train_fe_v1.csv"
TEST_OUTPUT_PATH = PROCEED_DIR / "test_fe_v1.csv"

TARGET = "Rings"
ID_COL = "id"
SEX_CATEGORIES = ["F", "I", "M"]

COLUMN_RENAME = {
    "Whole weight": "Whole_weight",
    "Whole weight.1": "Shucked_weight",
    "Whole weight.2": "Viscera_weight",
    "Shell weight": "Shell_weight",
}

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.6f}".format)


## 1. Load Raw Data

원본 데이터는 수정하지 않고 읽기만 합니다. train에는 target `Rings`가 있고 test에는 없습니다.


In [2]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

print(f"train_raw: {train_raw.shape}")
print(f"test_raw : {test_raw.shape}")

display(train_raw.head())
display(test_raw.head())


train_raw: (90615, 10)
test_raw : (60411, 9)


,id,Sex,Length,Diameter,Height,Whole weight,Whole weight.1,Whole weight.2,Shell weight,Rings
0,0,F,0.550000,0.430000,0.150000,0.771500,0.328500,0.146500,0.240000,11
1,1,F,0.630000,0.490000,0.145000,1.130000,0.458000,0.276500,0.320000,11
2,2,I,0.160000,0.110000,0.025000,0.021000,0.005500,0.003000,0.005000,6
3,3,M,0.595000,0.475000,0.150000,0.914500,0.375500,0.205500,0.250000,10
4,4,I,0.555000,0.425000,0.130000,0.782000,0.369500,0.160000,0.197500,9


,id,Sex,Length,Diameter,Height,Whole weight,Whole weight.1,Whole weight.2,Shell weight
0,90615,M,0.645000,0.475000,0.155000,1.238000,0.618500,0.312500,0.300500
1,90616,M,0.580000,0.460000,0.160000,0.983000,0.478500,0.219500,0.275000
2,90617,M,0.560000,0.420000,0.140000,0.839500,0.352500,0.184500,0.240500
3,90618,M,0.570000,0.490000,0.145000,0.874000,0.352500,0.186500,0.235000
4,90619,I,0.415000,0.325000,0.110000,0.358000,0.157500,0.067000,0.105000


## 2. Basic Validation

피처 생성 전에 원본 데이터의 기본 가정을 확인합니다. 실패하면 이후 산출물을 저장하지 않고 원인을 먼저 수정해야 합니다.


In [3]:
def validate_raw_inputs(train_df: pd.DataFrame, test_df: pd.DataFrame) -> None:
    assert ID_COL in train_df.columns and ID_COL in test_df.columns, "id column is required."
    assert TARGET in train_df.columns, "target column is required in train."
    assert TARGET not in test_df.columns, "target column should not exist in test."
    assert train_df[ID_COL].is_unique, "train id must be unique."
    assert test_df[ID_COL].is_unique, "test id must be unique."
    assert len(set(train_df[ID_COL]).intersection(test_df[ID_COL])) == 0, "train/test ids must not overlap."
    assert train_df.isna().sum().sum() == 0, "train has missing values."
    assert test_df.isna().sum().sum() == 0, "test has missing values."

validate_raw_inputs(train_raw, test_raw)

raw_check = pd.DataFrame(
    {
        "train": [len(train_raw), train_raw.shape[1], train_raw.duplicated().sum(), train_raw.isna().sum().sum()],
        "test": [len(test_raw), test_raw.shape[1], test_raw.duplicated().sum(), test_raw.isna().sum().sum()],
    },
    index=["rows", "columns", "duplicated_rows", "missing_cells"],
)

display(raw_check)


,train,test
rows,90615,60411
columns,10,9
duplicated_rows,0,0
missing_cells,0,0


## 3. Rename Columns

Kaggle CSV의 `Whole weight.1`, `Whole weight.2`는 각각 shucked weight, viscera weight에 해당합니다. 모델 입력과 해석을 위해 의미 있는 이름으로 바꿉니다.


In [4]:
train = train_raw.rename(columns=COLUMN_RENAME).copy()
test = test_raw.rename(columns=COLUMN_RENAME).copy()

base_numeric_features = [
    "Length",
    "Diameter",
    "Height",
    "Whole_weight",
    "Shucked_weight",
    "Viscera_weight",
    "Shell_weight",
]

expected_train_cols = [ID_COL, "Sex", *base_numeric_features, TARGET]
expected_test_cols = [ID_COL, "Sex", *base_numeric_features]

assert train.columns.tolist() == expected_train_cols, "Unexpected train schema after rename."
assert test.columns.tolist() == expected_test_cols, "Unexpected test schema after rename."

display(train.head())


,id,Sex,Length,Diameter,Height,Whole_weight,Shucked_weight,Viscera_weight,Shell_weight,Rings
0,0,F,0.550000,0.430000,0.150000,0.771500,0.328500,0.146500,0.240000,11
1,1,F,0.630000,0.490000,0.145000,1.130000,0.458000,0.276500,0.320000,11
2,2,I,0.160000,0.110000,0.025000,0.021000,0.005500,0.003000,0.005000,6
3,3,M,0.595000,0.475000,0.150000,0.914500,0.375500,0.205500,0.250000,10
4,4,I,0.555000,0.425000,0.130000,0.782000,0.369500,0.160000,0.197500,9


## 4. Feature Functions

비율 피처는 분모가 0이면 결측으로 둔 뒤, train 기준 median으로 채웁니다. 이렇게 하면 test 정보를 사용하지 않고 baseline 모델이 바로 읽을 수 있는 CSV가 됩니다.


In [5]:
def safe_divide(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    return numerator / denominator.replace(0, np.nan)


def add_domain_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["Height_is_zero"] = out["Height"].eq(0).astype(int)
    out["Volume"] = out["Length"] * out["Diameter"] * out["Height"]
    out["Density"] = safe_divide(out["Whole_weight"], out["Volume"])
    out["Shucked_ratio"] = safe_divide(out["Shucked_weight"], out["Whole_weight"])
    out["Viscera_ratio"] = safe_divide(out["Viscera_weight"], out["Whole_weight"])
    out["Shell_ratio"] = safe_divide(out["Shell_weight"], out["Whole_weight"])
    out["Shell_to_shucked"] = safe_divide(out["Shell_weight"], out["Shucked_weight"])
    return out


def add_sex_one_hot(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    sex_dummies = pd.get_dummies(out["Sex"], prefix="Sex", dtype=int)
    sex_dummies = sex_dummies.reindex(columns=[f"Sex_{value}" for value in SEX_CATEGORIES], fill_value=0)
    out = pd.concat([out.drop(columns=["Sex"]), sex_dummies], axis=1)
    return out


def replace_inf_with_nan(df: pd.DataFrame) -> pd.DataFrame:
    return df.replace([np.inf, -np.inf], np.nan)


## 5. Build v1 Feature Set

v1은 원본 수치 피처를 유지하고, EDA에서 확인한 안전한 파생 피처와 one-hot 피처를 추가합니다.


In [6]:
train_fe = add_sex_one_hot(add_domain_features(train))
test_fe = add_sex_one_hot(add_domain_features(test))

train_fe = replace_inf_with_nan(train_fe)
test_fe = replace_inf_with_nan(test_fe)

engineered_features = [
    "Height_is_zero",
    "Volume",
    "Density",
    "Shucked_ratio",
    "Viscera_ratio",
    "Shell_ratio",
    "Shell_to_shucked",
]
sex_features = [f"Sex_{value}" for value in SEX_CATEGORIES]
feature_cols = [*base_numeric_features, *engineered_features, *sex_features]

median_fill_values = train_fe[feature_cols].median(numeric_only=True)
train_fe[feature_cols] = train_fe[feature_cols].fillna(median_fill_values)
test_fe[feature_cols] = test_fe[feature_cols].fillna(median_fill_values)

train_fe = train_fe[[ID_COL, *feature_cols, TARGET]]
test_fe = test_fe[[ID_COL, *feature_cols]]

print(f"train_fe: {train_fe.shape}")
print(f"test_fe : {test_fe.shape}")

display(train_fe.head())
display(test_fe.head())


train_fe: (90615, 19)
test_fe : (60411, 18)


,id,Length,Diameter,Height,Whole_weight,Shucked_weight,Viscera_weight,Shell_weight,Height_is_zero,Volume,Density,Shucked_ratio,Viscera_ratio,Shell_ratio,Shell_to_shucked,Sex_F,Sex_I,Sex_M,Rings
0,0,0.550000,0.430000,0.150000,0.771500,0.328500,0.146500,0.240000,0,0.035475,21.747710,0.425794,0.189890,0.311082,0.730594,1,0,0,11
1,1,0.630000,0.490000,0.145000,1.130000,0.458000,0.276500,0.320000,0,0.044761,25.244909,0.405310,0.244690,0.283186,0.698690,1,0,0,11
2,2,0.160000,0.110000,0.025000,0.021000,0.005500,0.003000,0.005000,0,0.000440,47.727273,0.261905,0.142857,0.238095,0.909091,0,1,0,6
3,3,0.595000,0.475000,0.150000,0.914500,0.375500,0.205500,0.250000,0,0.042394,21.571576,0.410607,0.224713,0.273373,0.665779,0,0,1,10
4,4,0.555000,0.425000,0.130000,0.782000,0.369500,0.160000,0.197500,0,0.030664,25.502426,0.472506,0.204604,0.252558,0.534506,0,1,0,9


,id,Length,Diameter,Height,Whole_weight,Shucked_weight,Viscera_weight,Shell_weight,Height_is_zero,Volume,Density,Shucked_ratio,Viscera_ratio,Shell_ratio,Shell_to_shucked,Sex_F,Sex_I,Sex_M
0,90615,0.645000,0.475000,0.155000,1.238000,0.618500,0.312500,0.300500,0,0.047488,26.069675,0.499596,0.252423,0.242730,0.485853,0,0,1
1,90616,0.580000,0.460000,0.160000,0.983000,0.478500,0.219500,0.275000,0,0.042688,23.027549,0.486775,0.223296,0.279756,0.574713,0,0,1
2,90617,0.560000,0.420000,0.140000,0.839500,0.352500,0.184500,0.240500,0,0.032928,25.495019,0.419893,0.219774,0.286480,0.682270,0,0,1
3,90618,0.570000,0.490000,0.145000,0.874000,0.352500,0.186500,0.235000,0,0.040498,21.581046,0.403318,0.213387,0.268879,0.666667,0,0,1
4,90619,0.415000,0.325000,0.110000,0.358000,0.157500,0.067000,0.105000,0,0.014836,24.130087,0.439944,0.187151,0.293296,0.666667,0,1,0


## 6. Output Validation

저장 전에 train/test feature 컬럼이 완전히 일치하는지, 결측/무한대/문자형 컬럼이 남지 않았는지 확인합니다.


In [7]:
def validate_feature_outputs(train_df: pd.DataFrame, test_df: pd.DataFrame) -> None:
    train_feature_cols = [c for c in train_df.columns if c not in [ID_COL, TARGET]]
    test_feature_cols = [c for c in test_df.columns if c != ID_COL]

    assert train_feature_cols == test_feature_cols, "train/test feature columns differ."
    assert train_df[ID_COL].is_unique and test_df[ID_COL].is_unique, "id must remain unique."
    assert train_df[TARGET].equals(train_raw[TARGET]), "target values changed unexpectedly."
    assert train_df.isna().sum().sum() == 0, "train feature output has missing values."
    assert test_df.isna().sum().sum() == 0, "test feature output has missing values."
    assert np.isfinite(train_df[train_feature_cols + [TARGET]].to_numpy()).all(), "train output has non-finite values."
    assert np.isfinite(test_df[test_feature_cols].to_numpy()).all(), "test output has non-finite values."
    assert not train_df.drop(columns=[ID_COL]).select_dtypes(include="object").columns.any(), "train has object feature columns."
    assert not test_df.drop(columns=[ID_COL]).select_dtypes(include="object").columns.any(), "test has object feature columns."

validate_feature_outputs(train_fe, test_fe)

validation_summary = pd.DataFrame(
    {
        "train_fe": [train_fe.shape[0], train_fe.shape[1], train_fe.isna().sum().sum()],
        "test_fe": [test_fe.shape[0], test_fe.shape[1], test_fe.isna().sum().sum()],
    },
    index=["rows", "columns", "missing_cells"],
)

display(validation_summary)
display(train_fe[feature_cols + [TARGET]].describe().T)


,train_fe,test_fe
rows,90615,60411
columns,19,18
missing_cells,0,0


,count,mean,std,min,25%,50%,75%,max
Length,90615.000000,0.517098,0.118217,0.075000,0.445000,0.545000,0.600000,0.815000
Diameter,90615.000000,0.401679,0.098026,0.055000,0.345000,0.425000,0.470000,0.650000
Height,90615.000000,0.135464,0.038008,0.000000,0.110000,0.140000,0.160000,1.130000
Whole_weight,90615.000000,0.789035,0.457671,0.002000,0.419000,0.799500,1.067500,2.825500
Shucked_weight,90615.000000,0.340778,0.204428,0.001000,0.177500,0.330000,0.463000,1.488000
Viscera_weight,90615.000000,0.169422,0.100909,0.000500,0.086500,0.166000,0.232500,0.760000
Shell_weight,90615.000000,0.225898,0.130203,0.001500,0.120000,0.225000,0.305000,1.005000
Height_is_zero,90615.000000,0.000066,0.008137,0.000000,0.000000,0.000000,0.000000,1.000000
Volume,90615.000000,0.032850,0.019668,0.000000,0.016920,0.032340,0.044761,0.297472
Density,90615.000000,24.695522,4.027255,2.621073,22.583732,24.328358,26.281112,495.356037


## 7. Save Feature Files

`data/proceed`는 모델링 노트북이 재사용할 feature dataset 저장소입니다. v1 산출물은 아래 두 CSV입니다.


In [8]:
train_fe.to_csv(TRAIN_OUTPUT_PATH, index=False)
test_fe.to_csv(TEST_OUTPUT_PATH, index=False)

print(f"Saved: {TRAIN_OUTPUT_PATH.relative_to(PROJECT_ROOT)}")
print(f"Saved: {TEST_OUTPUT_PATH.relative_to(PROJECT_ROOT)}")
print(f"train rows: {len(train_fe):,}, test rows: {len(test_fe):,}")
print(f"feature count: {len(feature_cols):,}")


Saved: data/proceed/train_fe_v1.csv
Saved: data/proceed/test_fe_v1.csv
train rows: 90,615, test rows: 60,411
feature count: 17


## 8. v1 Feature Notes

생성된 v1 피처셋은 baseline을 위한 안정적인 시작점입니다.

- 입력 피처 수: 원본 수치 7개 + 파생 7개 + `Sex` one-hot 3개 = 17개
- train 출력: `id` + 17개 피처 + `Rings`
- test 출력: `id` + 17개 피처
- `Density`처럼 `Height == 0`에서 결측이 생기는 피처는 train median으로 채웠습니다.
- 이상치 제거는 아직 하지 않았습니다. 다음 baseline 단계에서 제거/클리핑/무처리를 CV로 비교하는 것이 좋습니다.
